# NULL Wallet: run everything from Jupyter
Put this notebook **inside the `null-wallet/` folder** (next to `onchain_demo.py`), open it in Jupyter, and run the cells top to bottom with Shift+Enter.
The private key is typed into a hidden prompt and is never saved in this notebook. Do not share the notebook after running Part 5 without clearing its outputs (Kernel > Restart & Clear Output).

## 0. Setup

In [ ]:
import os, sys, subprocess, shlex, getpass, pathlib
assert pathlib.Path('onchain_demo.py').exists(), 'Move this notebook into the null-wallet/ folder first'
print('Python', sys.version.split()[0], '|', sys.executable)
def run(cmd):
    """Run a command with the same Python as this notebook and stream its output (works on Windows, Mac and Linux)."""
    print('$', cmd)
    parts = [p.strip('"\'') for p in shlex.split(cmd, posix=(os.name != 'nt'))]
    if parts[0] == 'python': parts[0] = sys.executable
    env = os.environ.copy(); env['PYTHONUTF8'] = '1'; env['PYTHONIOENCODING'] = 'utf-8'
    p = subprocess.Popen(parts, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', env=env)
    for line in p.stdout: print(line, end='')
    p.wait(); print('exit code', p.returncode); return p.returncode
run('python -m pip install -r requirements.txt')

## 1. Tests (expect `OK`, 51 tests)

In [ ]:
run('python -m unittest discover -s tests')

## 2. Your BTC data (csv files)
Set the two paths, set `QUICK = True` first for a 5-minute smoke test, then `False` for the full run. It writes `results_for_claude.zip`.

In [ ]:
TRAIN = 'path/to/btc_train.csv'
TEST  = 'path/to/btc_test.csv'
QUICK = True
run(f'python run_all.py {TRAIN} {TEST}' + (' --quick' if QUICK else ''))

## 3. Public Binance data (outside the course)
Read Binance's data terms first (data.binance.vision). Do not commit the downloaded csv.

In [ ]:
os.makedirs('data', exist_ok=True)
run('python fetch_data.py BTCUSDT 2023-01 2023-12 data/btc23.csv')
run('python run_evidence.py data/btc23.csv results/btc23')

Send me `results_for_claude.zip` and `results/btc23/summary.json`.

## 4. On-chain evidence on Base mainnet (about 0.002 ETH in a brand-new wallet)
**No MetaMask needed.** Run the next cell to create a throwaway wallet on your own computer. It prints an ADDRESS and saves the key in `throwaway_wallet.key`.
Then send about 0.002 ETH to that address from an exchange or app available to you, choosing the **Base** network when you withdraw. Wait until the balance shows at basescan.org/address/<your address> before running the dry run.

In [ ]:
run('python new_wallet.py')

In [ ]:
# several public Base RPC URLs: if one refuses (HTTP 403), the next is tried automatically
os.environ['NULLWALLET_RPC'] = 'https://mainnet.base.org,https://base-rpc.publicnode.com,https://1rpc.io/base,https://base.llamarpc.com'
os.environ['NULLWALLET_KEY'] = (open('throwaway_wallet.key').read().strip() if os.path.exists('throwaway_wallet.key')
    else getpass.getpass('Throwaway private key (64 hex characters, no 0x): '))
run('python onchain_demo.py results/ledger_1h.jsonl')   # DRY RUN: sends nothing; look for: chain 8453 (REAL NETWORK)

In [ ]:
sys.path.insert(0, '.')
from nullwallet import onchain
rpc = onchain.Rpc(os.environ['NULLWALLET_RPC']); me = onchain.Signer()
print('wallet', me.address, '| using', rpc.url)
print('chain id', rpc.chain_id(), '(must be 8453) | ETH balance on Base:', rpc.balance(me.address) / 1e18, '(need at least 0.0008)')

In [ ]:
# REAL RUN: only after the dry run looks right and the wallet holds about 0.002 ETH on Base.
CONFIRM_SEND = False   # change to True to broadcast
if CONFIRM_SEND:
    run('python onchain_demo.py results/ledger_1h.jsonl --send')
else:
    print('Not sending. Set CONFIRM_SEND = True and re-run this cell.')

In [ ]:
import json
ev = json.load(open('evidence/onchain_evidence.json'))
print(json.dumps(ev, indent=2)[:3000])   # open every link; Input Data must start with 0x4e554c4c31

Send me `evidence/onchain_evidence.json`. Afterwards: restart the kernel, clear outputs, and delete the throwaway key.